### Input Libraries

In [140]:
import json
import re
from langchain.schema import BaseOutputParser
from langchain.chains.router.llm_router import LLMRouterChain
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama
import math

### Define LLM

In [141]:
# Specify the remote server's URL
# llm = Ollama(model="deepseek-r1:1.5b-qwen-distill-q4_K_M", base_url="http://127.0.0.1:11434")
llm = Ollama(model="phi4:latest", base_url="http://127.0.0.1:11434")


#### Using LLM chains for extracting parameters 

In [142]:
# import re
# import json
# from langchain.chains import LLMChain
# from langchain.prompts import PromptTemplate
# from langchain_community.llms import Ollama


# extraction_prompt = PromptTemplate(
#     input_variables=["user_input"],
#     template=(
#         "User query:\n"
#         "\"{user_input}\"\n\n"
#         "Extract these fields as *pure JSON* (no markdown fences, no extra text):\n"
#         "- deposit_amount (integer months: deposit)\n"
#         "- loan_amount (integer montoan)\n"
#         "- deposit_duration (integer months)\n"
#         "- repayment_duration (integer months)\n"
#         "- Credit_score (string or null)\n"
#         "- Interest_rate (integer percent without % sign)\n\n"
#         "If missing, set value to null.\n"
#         "Numbers must be plain digits (e.g. 25000000), no underscores or commas.\n\n"
#         "Example output:\n"
#         "{{\n"
#         '  "deposit_amount": 500000000,\n'
#         '  "deposit_duration": 3,\n'
#         '  "loan_amount": 150000000,\n'
#         '  "repayment_duration": 36,\n'
#         '  "Credit_score": "B",\n'
#         '  "Interest_rate": 23\n'
#         "}}\n"
#     )
# )


# extraction_chain = LLMChain(
#     llm=llm,
#     prompt=extraction_prompt,
#     verbose=False,
# )

# # 3) Helper to clean & load JSON
# def clean_and_parse(raw: str) -> dict:
#     # strip markdown fences if any
#     cleaned = re.sub(r"```(?:json)?\s*", "", raw)
#     cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
#     # isolate the braces & their contents
#     match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
#     if not match:
#         raise ValueError(f"No JSON object found in LLM output:\n{repr(raw)}")
#     js = match.group(0)
#     # remove illegal underscores in numbers
#     js = re.sub(r"(?<=\d)_(?=\d)", "", js)
#     # remove trailing commas before closing brace/bracket
#     js = re.sub(r",\s*([\}\]])", r"\1", js)
#     # strip out any percent signs
#     js = js.replace("%", "")
#     # finally parse
#     return json.loads(js)

# # 4) Run end-to-end
# user_input = "من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره"
# raw_output = extraction_chain.predict(user_input=user_input)
# print("Raw LLM output:\n", raw_output)

# try:
#     params = clean_and_parse(raw_output)
#     print("\nExtracted parameters:", params)
# except Exception as e:
#     print("❌ Parsing failed:", e)


In [143]:
from langchain.prompts import PromptTemplate

        # "- deposit_amount (integer or null) \n"
        # "- loan_amount (integer or null)\n"
        # "- deposit_duration (integer months or null)\n"
        # "- repayment_duration (integer months or null)\n"
        # "- Credit_score (string or null)\n"
        # "- Interest_rate (integer percent without % or null)\n\n"


extraction_prompt = PromptTemplate(
    input_variables=["user_input"],
    template=(
        # 1) absolutely forbid chain‐of‐thought
        "Do not think. Do not output any reasoning—output **only** the JSON.\n\n"

        # 2) define the fields
        "Extract exactly these fields as JSON (no markdown, no fences):\n"
        "- deposit_amount (float or null) : مقدار سپرده یا میانگین سپرده یا میزان پولی که کاربر دارد \n"
        "- loan_amount (float or null) : مقدار وامی که کاربر میخواهد یا به او تعلق میگیرد\n"
        "- deposit_duration (integer months or null) : مدت زمانی که پول یا سپرده مشتری در حساب بانکی باید باشد یا میخواهد باش یا در حسابش بخواباند\n"
        "- repayment_duration (integer months or null) : مدت زمان بازپرداخت وام یا تعداد اقساط\n"
        "- Credit_score (string or null) : امتیاز اعتباری یا رتبه اعتباری\n"
        "- Interest_rate (integer percent without % or null) : نرخ سود وام یا کارمزد وام\n\n"
        "مقدار سپرده و مقدار وام به صورت پیش فرض بر حسب تومان هستند. اگر کاربر به تومن مقادیر را بیان کرد آن را با فرض ریال بودن درنظر بگیر\n"
        # 3) if a field is not mentioned, it must be null
        "If missing, set its value to `null`. Numbers must be plain digits (e.g. 25000000),\n"
        "no underscores, commas, or % signs.\n\n"

        # 4) few‐shot example for missing durations
        "### Example 1\n"
        "Input: \"من میخوام ببینم سود سپرده بانک اگه ۶۰ میلیون پول بخوابونم چقدره\"\n"
        "Output:\n"
        '{{'
        '"deposit_amount":600_000_000,'
        '"deposit_duration":null,'
        '"loan_amount":null,'
        '"repayment_duration":null,'
        '"Credit_score":null,'
        '"Interest_rate":null'
        '}}\n\n'

        # 5) now YOUR input
        "### Now process this input:\n"
        "Input: \"{user_input}\"\n"
        "Output:"
    )
)


In [144]:
import re
import json

def clean_and_parse(raw: str) -> dict:
    # 1) Strip any markdown fences (``` or ```json)
    cleaned = re.sub(r"```(?:json)?\s*\n?", "", raw)
    cleaned = re.sub(r"\n?```", "", cleaned)

    # 2) Now isolate the first {...} block in the remaining text
    match = re.search(r"\{[\s\S]*?\}", cleaned)
    if not match:
        raise ValueError(f"No JSON object found in LLM output:\n{raw!r}")
    js = match.group(0)

    # 3) Cleanup common issues:
    #    - trailing commas before a closing brace/bracket
    #    - percent signs
    #    - underscores in numbers
    js = re.sub(r",\s*([\}\]])", r"\1", js)
    js = js.replace("%", "")
    js = re.sub(r"(?<=\d)_(?=\d)", "", js)

    # 4) Finally, parse
    return json.loads(js)


In [150]:

extraction_chain = LLMChain(
    llm=llm,
    prompt=extraction_prompt,
    verbose=False,
)

# user_input = "من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره"
# user_input = "با ۳۵ میلیون تومن چقدر وام ۴ درصد میتونم بگیرم"
# user_input = "من یه وام ۲۰۰ میلیونیه ۲ ساله میخوام"
# user_input = "من الان ۳۰ میلیون پول تو حسابم دارم. چقدر تو حسابم باشه این  پول که بتونم ۱۰۰ میلیون وام ۴ درصد بگیرم"
# user_input = "وام ۲۳ درصد که اقساطش ۱۲ ماهه باشه چی داری. دور و ور ۴۰، ۵۰ میلیون وام میخوام"
user_input = "یه وام ۱۰۰ تومنی میخوام ۲ ساله. اگه ۱۵۰ تومن بزارم تو حسابم بهم تعلق میگیره"

raw = extraction_chain.predict(user_input=user_input)
print("🔴 Raw LLM output:\n", raw)

try:
    params = clean_and_parse(raw)
    print("✅ Extracted parameters:", params)
except Exception as e:
    print("❌ Parsing failed:", e)


🔴 Raw LLM output:
 ```json
{
    "deposit_amount": 150_000_000,
    "loan_amount": 100_000_000,
    "deposit_duration": null,
    "repayment_duration": 24,
    "Credit_score": null,
    "Interest_rate": null
}
```
✅ Extracted parameters: {'deposit_amount': 150000000, 'loan_amount': 100000000, 'deposit_duration': None, 'repayment_duration': 24, 'Credit_score': None, 'Interest_rate': None}


In [ ]:
100_000_000

100000000